# Implicit Decision Gate

## Motivation

Long-running AI work can quietly make important choices that the original request never made. A request to add an export feature might not say who may export, how long exported files should be kept, or whether each export must be recorded. The code must still choose a behavior, and that choice can be hard to notice inside a large change.

The larger idea behind this project is one shared gate for these missing decisions. Separate checks for important parts of a system report simple facts about what the agent actually changed. A database check can report what happens to existing data, a permission check can report who gained access, a storage check can report how long data is kept, and an API check can report behavior visible to other software. If a reported fact matters and the request contains no approved answer for it, the gate saves the work and asks a person.

This scales by building each kind of check once and reusing it across many jobs. The shared gate handles saving, asking, resuming, and checking the next result for all of them. It doesn't promise to find every possible hidden choice. It covers important parts of a system where effects can be observed reliably.

## Premise

Implicit Decision Gate is a deliberately small, fictional contract-completion stage inside the trust architecture described in 1Password's [Verified Loops](https://1password.com/blog/verified-loops-building-ai-agent-trust). That architecture makes the human-owned job definition the verification boundary and leaves humans the consequential judgments that can't be verified mechanically.

Imagine a workspace export service. Its brief specifies two behaviors:

| Brief specifies | Brief doesn't specify |
| --- | --- |
| The first owner request creates an export | Whether administrators can create exports |
| Members are denied | What a repeated owner request should do |

The generated handler must still choose both unspecified behaviors. Any supported combination could be legitimate, but the coding agent shouldn't silently make those product decisions. This walkthrough shows the gate observing both choices, requesting both answers in one durable pause, and verifying both after one fresh retry.

The scenario is fictional and makes no claim about 1Password's production services or authorization model.

## System context

The gate sits between generated work and the wider verified loop. It doesn't supply identity, tool controls, or permission enforcement. It completes missing intent from observed effects and returns a verified result.

![System context showing the human brief, coding process, behavior observer, evidence reviewer, durable gate, and wider verified loop.](assets/diagrams/system_context.png)

[Review the Mermaid source.](assets/diagrams/system_context.mmd)

## What this walkthrough proves

- One observer can report multiple independent decisions from one generated artifact.
- The gate can collect all required human answers in one durable pause.
- One fresh coding attempt can be checked against the complete decision set.

![Lifecycle showing two observed decisions, one durable pause, two human answers, one fresh retry, and final verification.](assets/diagrams/lifecycle.png)

[Review the Mermaid source.](assets/diagrams/lifecycle.mmd)

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "examples/workspace-export-authorization/brief.md"
        ).is_file():
            return candidate.resolve()
    raise RuntimeError("Open this notebook from inside the repository")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
COMMAND_ENV = os.environ.copy()
COMMAND_ENV.pop("VIRTUAL_ENV", None)
SCENARIO = "workspace-export-authorization"
BRIEF_PATH = "examples/workspace-export-authorization/brief.md"
ADMINISTRATOR_ACCESS = "workspace_export_administrator_access"
REPEAT_REQUEST = "workspace_export_repeat_request"
DECISION_ORDER = (ADMINISTRATOR_ACCESS, REPEAT_REQUEST)
DECISION_NAMES = {
    ADMINISTRATOR_ACCESS: "Administrator access",
    REPEAT_REQUEST: "Repeated owner request",
}


def run_idg(*arguments: str) -> dict[str, Any]:
    completed = subprocess.run(
        ["uv", "run", "idg", *arguments],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
        env=COMMAND_ENV,
    )
    if not completed.stdout:
        raise RuntimeError(completed.stderr.strip() or "idg returned no JSON")
    payload = json.loads(completed.stdout)
    if completed.returncode not in (0, 1):
        raise RuntimeError(completed.stderr.strip() or str(payload))
    return payload


def read_at_head(path: str) -> str:
    return subprocess.run(
        ["git", "show", f"HEAD:{path}"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout


def load_run(run_id: str) -> dict[str, Any]:
    path = REPO_ROOT / ".idg" / "runs" / run_id / "run.json"
    return json.loads(path.read_text(encoding="utf-8"))


def show_table(headers: tuple[str, ...], rows: list[tuple[str, ...]]) -> None:
    header = "| " + " | ".join(headers) + " |"
    divider = "| " + " | ".join("---" for _ in headers) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    display(Markdown("\n".join([header, divider, *body])))

## First attempt

The application pins the brief and baseline handler to the current Git commit, asks a fresh Codex process to implement the handler, and executes the result in a disposable, network-disabled container. This cell invokes the live Codex CLI.

In [ ]:
BRIEF = read_at_head(BRIEF_PATH).strip()
display(Markdown(f"```text\n{BRIEF}\n```"))

START_SUMMARY = run_idg("start", "--scenario", SCENARIO)
RUN_ID = str(START_SUMMARY["run_id"])
START_STATE = START_SUMMARY["state"]
PRODUCT_FLOW = START_STATE == "AWAITING_OWNER"
COVERAGE_GAPS = START_SUMMARY["coverage_gaps"]
if PRODUCT_FLOW:
    display(Markdown(f"Run state: `{START_STATE}`"))
elif START_STATE == "COVERAGE_GAP":
    if not COVERAGE_GAPS:
        raise RuntimeError("COVERAGE_GAP returned without a persisted event")
    display(Markdown(f"Run state: `{START_STATE}`"))
    coverage_rows = []
    for gap in COVERAGE_GAPS:
        facts = "<br>".join(f"`{key}={json.dumps(value)}`" for key, value in gap["facts"].items())
        coverage_rows.append(
            (
                DECISION_NAMES.get(gap["decision_id"], gap["decision_id"]),
                f"`{gap['observed']}`",
                facts,
            )
        )
    show_table(("Coverage gap", "Observed outcome", "Normalized facts"), coverage_rows)
    display(
        Markdown(
            "The event is persisted for later platform engineering review. "
            "No product decision or retry was started."
        )
    )
else:
    raise RuntimeError(
        f"Expected AWAITING_OWNER or COVERAGE_GAP, found {START_STATE}: {START_SUMMARY['error']}"
    )

## The gate pauses

The observer calls the generated handler twice as an owner with shared state, then once each as an administrator and member. When both observed combinations match approved coverage, it reports two typed outcomes. A separate evidence review compares each outcome with the brief. Because the brief supplies neither answer, the gate presents both questions together. If a combination is unmodeled, the gate instead records a coverage event for later platform engineering review and stops before the product-decision flow. The diagram shows the modeled path.

<img src="assets/diagrams/gate_logic.png" alt="Gate logic showing the coverage-gap route, independent outcome review, one owner pause, and verification of every expected outcome." width="520">

[Review the Mermaid source.](assets/diagrams/gate_logic.mmd)

In [ ]:
REQUESTS = {request["id"]: request for request in START_SUMMARY["decision_requests"]}
if PRODUCT_FLOW:
    if set(REQUESTS) != set(DECISION_ORDER):
        raise RuntimeError("The live run didn't produce both expected decision requests")

    decision_rows = []
    for decision_id in DECISION_ORDER:
        request = REQUESTS[decision_id]
        options = " or ".join(f"`{option['option']}`" for option in request["options"])
        decision_rows.append(
            (
                request["question"],
                f"`{request['observed']['option']}`",
                f"`{START_SUMMARY['classifications'][decision_id]}`",
                options,
            )
        )

    show_table(
        ("Missing question", "Observed choice", "Evidence review", "Available answers"),
        decision_rows,
    )
else:
    display(Markdown("No product questions were created for this coverage event."))

## Human completes the contract

On the modeled path, review or edit the two values below. Each `answer` command records one typed decision without invoking a model. The run remains paused after the first answer and becomes ready only after the second answer. A coverage-gap run skips this section.

In [ ]:
if PRODUCT_FLOW:
    # Human input: edit either value before running this cell.
    OWNER_DECISIONS = {
        ADMINISTRATOR_ACCESS: "OWNER_ONLY",
        REPEAT_REQUEST: "REUSE_ACTIVE_EXPORT",
    }

    transition_rows = []
    for decision_id in DECISION_ORDER:
        option_id = OWNER_DECISIONS[decision_id]
        allowed = {option["option"] for option in REQUESTS[decision_id]["options"]}
        if option_id not in allowed:
            raise ValueError(f"{option_id} isn't valid for {decision_id}")
        answer_summary = run_idg("answer", RUN_ID, "--decision", decision_id, "--option", option_id)
        transition_rows.append(
            (DECISION_NAMES[decision_id], f"`{option_id}`", f"`{answer_summary['state']}`")
        )

    show_table(("Answer recorded", "Selected option", "Run state"), transition_rows)
    if answer_summary["state"] != "READY_TO_RESUME":
        raise RuntimeError(f"Expected READY_TO_RESUME, found {answer_summary['state']}")
else:
    display(Markdown("No product answers were requested or recorded."))

## Fresh retry and verification

On the modeled path, `resume` starts one new coding process from the original commit. It receives the original brief, baseline handler, and both owner decisions. It doesn't receive the first artifact or the reviewer rationale. The same observer then verifies every expected outcome. A coverage-gap run starts no retry.

In [ ]:
if PRODUCT_FLOW:
    RESUME_SUMMARY = run_idg("resume", RUN_ID)
    FINAL_SNAPSHOT = load_run(RUN_ID)
    attempts = FINAL_SNAPSHOT["attempts"]
    if len(attempts) != 2:
        raise RuntimeError(f"Expected two attempts, found {len(attempts)}")

    first_outcomes = attempts[0]["observation"]["outcomes"]
    second_outcomes = attempts[1]["observation"]["outcomes"]
    selected = {
        decision["decision_id"]: decision["selected"] for decision in FINAL_SNAPSHOT["decisions"]
    }
    verification_rows = []
    verified = 0
    for decision_id in DECISION_ORDER:
        matches = selected[decision_id] == second_outcomes[decision_id]
        verified += int(matches)
        verification_rows.append(
            (
                DECISION_NAMES[decision_id],
                f"`{first_outcomes[decision_id]}`",
                f"`{selected[decision_id]}`",
                f"`{second_outcomes[decision_id]}`",
                "Verified" if matches else "Mismatch",
            )
        )

    show_table(
        ("Decision", "First attempt", "Owner selected", "Second attempt", "Result"),
        verification_rows,
    )
    clean_retry = (
        attempts[1]["clean_start_verified"]
        and attempts[0]["worktree_path"] != attempts[1]["worktree_path"]
    )
    show_table(
        ("Measure", "Result"),
        [
            ("Missing decisions detected", str(len(START_SUMMARY["decision_requests"]))),
            ("Human answers recorded", str(len(selected))),
            ("Clean retries performed", "1" if clean_retry else "0"),
            ("Verified outcomes", f"{verified} of {len(DECISION_ORDER)}"),
            ("Final state", f"`{RESUME_SUMMARY['state']}`"),
        ],
    )
    if RESUME_SUMMARY["state"] != "COMPLETED":
        raise RuntimeError(RESUME_SUMMARY["error"] or "Verification failed")
else:
    display(Markdown("No retry was started because this run requires platform coverage review."))

## Why this matters

- The decision requests came from container-observed effects, not the coding agent's explanation.
- One behavioral surface exposed both an authorization choice and a repeated-request choice.
- The owner completed both missing parts of the contract before one new coding attempt.
- The gate checked both selected outcomes directly before completing the modeled run.
- An unmodeled result is preserved for later platform review without becoming a product question.

The same gate also accepts outcomes from the repository's PostgreSQL observer. Each new surface still needs a bounded observer and supported vocabulary, but saving, reviewing, pausing, answering, retrying, and verifying remain shared.